# Train on Kaggle

Written by `scripts/make_kaggle_notebook.py`. Before running:

1. Build the code bundle: `../tse_venv/bin/python scripts/make_kaggle_bundle.py --config <config>`, then upload `kaggle_bundle/kaggle_code.zip` as a new version of the code dataset.
2. Attach the code dataset and `tse-audio-s0-v3`. Settings: Accelerator GPU T4, Internet on (pip installs and the probe's Whisper model).
3. Run all, then download `ALL-*.zip` from the output panel.

In [ ]:
# The run, set once. RESUME_DIR is an attached earlier output whose models/ folder holds the checkpoints, to continue that run with --resume.
SCRIPT = "scripts/train_turn_down.py"
CONFIG = "experiments/configs/turn_down_head.yaml"
EPOCHS = 6
RESUME_DIR = None
DATA_DIR = "/kaggle/input/tse-audio-s0-v3"
CODE_DIR = "/kaggle/input/tse-code-v3"
WORK = "/kaggle/working"

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU. Set Settings, Accelerator, GPU T4 before training.")
print(f"torch {torch.__version__} on {torch.cuda.get_device_name(0)}")

# The bundle is read-only under /kaggle/input, so the repo and the ECAPA snapshot are copied side by side into the working folder.
code = Path(CODE_DIR)
work = Path(WORK)
repo = work / "repo"
for name in ["repo", "ecapa_pretrained"]:
    if (work / name).exists():
        shutil.rmtree(work / name)
    shutil.copytree(code / name, work / name)

manifest = json.loads((code / "bundle_manifest.json").read_text())
# The repo copy has no git history, so the training script reads the bundle's commit from here.
shutil.copy(code / "bundle_manifest.json", repo / "bundle_manifest.json")
print(f"bundle built {manifest['date']} from commit {manifest['git_commit']}, checked with {manifest['script']} and {manifest['config']}")
if manifest["config"] != CONFIG or manifest["script"] != SCRIPT:
    print(f"NOTE: the bundle was checked with {manifest['script']} and {manifest['config']}, but this run uses {SCRIPT} and {CONFIG}")
if not (repo / CONFIG).exists() or not (repo / SCRIPT).exists():
    raise SystemExit(f"{SCRIPT} or {CONFIG} is not in the bundle.")

# Pinned as in requirements.txt. speechbrain runs the ECAPA encoder; the other three run the per-epoch content probe.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "speechbrain==1.1.1", "faster-whisper==1.2.1", "jiwer==4.0.0", "whisper-normalizer==0.1.15", "soundfile"], check=True)

models = work / "models"
models.mkdir(exist_ok=True)
if RESUME_DIR:
    for checkpoint in sorted((Path(RESUME_DIR) / "models").glob("*.pt")):
        shutil.copy(checkpoint, models / checkpoint.name)
        print(f"resuming with {checkpoint.name} from {RESUME_DIR}")

In [ ]:
command = [sys.executable, "-u", SCRIPT, "--config", CONFIG, "--epochs", str(EPOCHS), "--data-root", f"{DATA_DIR}/data", "--manifest-dir", f"{DATA_DIR}/data/manifests", "--outdir", str(models)]
if RESUME_DIR:
    command.append("--resume")
print(" ".join(command), flush=True)
process = subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end="")
return_code = process.wait()
print(f"exit {return_code}")

In [ ]:
import zipfile

# One download holds the checkpoints, the results folder train.py wrote, the config that ran and the run-time log.
archive_path = work / f"ALL-{Path(CONFIG).stem}-e{EPOCHS}.zip"
results = repo / "experiments/results"
with zipfile.ZipFile(archive_path, "w", zipfile.ZIP_STORED) as archive:
    for path in sorted(models.glob("*.pt")):
        archive.write(path, f"models/{path.name}")
    if results.exists():
        for path in sorted(results.rglob("*")):
            if path.is_file():
                archive.write(path, f"results/{path.relative_to(results)}")
    archive.write(repo / CONFIG, Path(CONFIG).name)
    archive.write(repo / "docs/run_times.md", "run_times.md")
print(f"{archive_path}: {archive_path.stat().st_size / 1e6:.0f} MB")